# Helmet Detector Fine-tuning (YOLOv11n)

1. **Add Input** — attach your helmet Kaggle dataset
2. Run **Inspect dataset** cell — note the path under `/kaggle/input/`
3. Edit `DATASET_PATH` in the yaml cell if needed
4. Run training — Ultralytics saves plots to `runs/detect/helmet_train/`
5. Run **View training plots** to see loss/mAP/confusion matrix inline
6. Download `runs/detect/helmet_train/weights/best.pt` → `models/helmet_best.pt`

In [ ]:
!pip install -q ultralytics

In [ ]:
# Inspect attached Kaggle inputs — update DATASET_PATH below to match your folder name
import os
from pathlib import Path

for root, dirs, files in os.walk('/kaggle/input'):
    level = root.replace('/kaggle/input', '').count(os.sep)
    print(f"{'  ' * level}{Path(root).name}/  ({len(files)} files)")

# <-- Change this to your helmet dataset folder under /kaggle/input/
DATASET_PATH = '/kaggle/input'  # e.g. '/kaggle/input/myanmar-helmet-detection'
print('\nUsing DATASET_PATH:', DATASET_PATH)

In [ ]:
# Create YOLO data yaml (adjust train/val paths if your dataset layout differs)
yaml_content = f"""path: {DATASET_PATH}
train: images/train
val: images/val

names:
  0: helmet
  1: no_helmet
  2: motorcycle
"""
with open('/kaggle/working/helmet.yaml', 'w') as f:
    f.write(yaml_content)
print(yaml_content)

In [ ]:
from ultralytics import YOLO

RUN_NAME = 'helmet_train'

model = YOLO('yolo11n.pt')
results = model.train(
    data='/kaggle/working/helmet.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    save_period=10,
    name=RUN_NAME,
    plots=True,       # save training/validation plots (default, explicit here)
    save=True,
    verbose=True,
)
print('Training complete. Output dir:', results.save_dir)

In [ ]:
from ultralytics import YOLO

best = YOLO(f'runs/detect/{RUN_NAME}/weights/best.pt')
metrics = best.val(plots=True)  # also saves val plots (confusion matrix, PR curve, etc.)
print(f'mAP50: {metrics.box.map50:.3f}')
print(f'mAP50-95: {metrics.box.map50_95:.3f}')
print('Target: mAP50 > 0.85 — prioritize no_helmet recall')

In [ ]:
# View training plots inline (saved automatically by Ultralytics)
from pathlib import Path
from IPython.display import Image, display, Markdown
import pandas as pd
import matplotlib.pyplot as plt

RUN_DIR = Path(f'runs/detect/{RUN_NAME}')

# --- Static plots saved during training ---
plot_files = [
    ('Training metrics (loss, mAP, precision, recall)', 'results.png'),
    ('Confusion matrix', 'confusion_matrix.png'),
    ('Confusion matrix (normalized)', 'confusion_matrix_normalized.png'),
    ('Precision-Recall curve', 'PR_curve.png'),
    ('F1 curve', 'F1_curve.png'),
    ('Precision curve', 'P_curve.png'),
    ('Recall curve', 'R_curve.png'),
    ('Label distribution', 'labels.jpg'),
    ('Label correlogram', 'labels_correlogram.jpg'),
]

for title, filename in plot_files:
    path = RUN_DIR / filename
    if path.exists():
        display(Markdown(f'### {title}'))
        display(Image(filename=str(path), width=900))

# --- Custom curves from results.csv (more detailed per-epoch) ---
csv_path = RUN_DIR / 'results.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Helmet model — per-epoch training curves', fontsize=14)

    if 'train/box_loss' in df.columns:
        axes[0, 0].plot(df['epoch'], df['train/box_loss'], label='train box', color='#e74c3c')
        axes[0, 0].plot(df['epoch'], df['val/box_loss'], label='val box', color='#3498db')
        axes[0, 0].set_title('Box loss'); axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

    if 'metrics/mAP50(B)' in df.columns:
        axes[0, 1].plot(df['epoch'], df['metrics/mAP50(B)'], color='#2ecc71', linewidth=2)
        axes[0, 1].set_title('mAP50'); axes[0, 1].grid(True, alpha=0.3)

    if 'metrics/precision(B)' in df.columns:
        axes[1, 0].plot(df['epoch'], df['metrics/precision(B)'], label='precision', color='#9b59b6')
        axes[1, 0].plot(df['epoch'], df['metrics/recall(B)'], label='recall', color='#f39c12')
        axes[1, 0].set_title('Precision & Recall'); axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

    if 'train/cls_loss' in df.columns:
        axes[1, 1].plot(df['epoch'], df['train/cls_loss'], label='train cls', color='#e74c3c')
        axes[1, 1].plot(df['epoch'], df['val/cls_loss'], label='val cls', color='#3498db')
        axes[1, 1].set_title('Classification loss'); axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print('results.csv not found — run training first')